In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Dense, Flatten, Concatenate, Attention, Dropout, LayerNormalization
from tensorflow.keras.layers import Reshape

from data_preparation import data_preparation


## **1. Data Preparation**

In [38]:
X_train, X_val, X_test, y_train, y_val, y_test = data_preparation('../dataset/digital_marketing_campaign_dataset.csv')

## **2. Preprocessing**

In [39]:
# 1. Encode Categorical Feature
le = LabelEncoder()
X_train['CampaignType'] = le.fit_transform(X_train['CampaignType'])
X_val['CampaignType'] = le.transform(X_val['CampaignType'])
X_test['CampaignType'] = le.transform(X_test['CampaignType'])

# 2. Scale Numerical Features
num_features = ['AdSpend', 'ClickThroughRate', 'WebsiteVisits', 'PagesPerVisit', 
                'TimeOnSite', 'EmailOpens', 'EmailClicks', 'PreviousPurchases', 'LoyaltyPoints']

scaler = StandardScaler()
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_val[num_features] = scaler.transform(X_val[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

# Tách input cho model (Categorical và Numerical riêng biệt)
def get_inputs(X):
    return [X['CampaignType'].values, X[num_features].values]

X_train_in = get_inputs(X_train)
X_val_in = get_inputs(X_val)
X_test_in = get_inputs(X_test)

## **3. Hyperparamter Configuration**

In [40]:
VOCAB_SIZE = len(le.classes_)  
EMBED_DIM = 16                 
HIDDEN_UNITS = [64, 32]        
DROPOUT_RATE = 0.4             

LEARNING_RATE = 0.0005          
BATCH_SIZE = 64               
EPOCHS = 100                    

## **4. Training**

In [41]:
cat_input = Input(shape=(1,), name='cat_input')
num_input = Input(shape=(len(num_features),), name='num_input')

# Embedding
embed = Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)(cat_input)
embed = Flatten()(embed)

# Concatenate & Attention 
combined = Concatenate()([embed, num_input])


query_seq = Reshape((1, -1))(combined) 
attention_out = Flatten()(Attention()([query_seq, query_seq]))

# Deep Layers
x = Dense(HIDDEN_UNITS[0], activation='relu')(attention_out)
x = Dropout(DROPOUT_RATE)(x)
x = Dense(HIDDEN_UNITS[1], activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=[cat_input, num_input], outputs=output)


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Early Stopping để lấy model tốt nhất
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train_in, y_train,
    validation_data=(X_val_in, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100


c:\Users\TaiBui\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\ops\nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (64, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8330 - auc: 0.5042 - loss: 0.4764 - val_accuracy: 0.8766 - val_auc: 0.5541 - val_loss: 0.3807
Epoch 2/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8768 - auc: 0.6127 - loss: 0.3710 - val_accuracy: 0.8766 - val_auc: 0.6994 - val_loss: 0.3448
Epoch 3/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8773 - auc: 0.6930 - loss: 0.3500 - val_accuracy: 0.8773 - val_auc: 0.7475 - val_loss: 0.3281
Epoch 4/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8799 - auc: 0.7335 - loss: 0.3332 - val_accuracy: 0.8836 - val_auc: 0.7685 - val_loss: 0.3186
Epoch 5/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8803 - auc: 0.7460 - loss: 0.3280 - val_accuracy: 0.8820 - val_auc: 0.7770 - val_loss: 0.3138
Epoch 6/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8834 - auc: 0.7432 - loss: 0.3299 - val_accuracy: 0.8820 - val_auc: 0.7799 - val_loss: 0.3114
Epoch 7/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accur

## **5. Predict testing dataset**

In [ ]:
y_pred_prob = model.predict(X_test_in)

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


In [ ]:
# 1. Tạo DataFrame tổng hợp
comparison_df = pd.DataFrame({
    'Actual': y_test.values,
    'embedding_attention_mlp': y_prob_mlp.flatten(),

})

# 2. Tính nhãn dự đoán (0/1) cho từng cái
comparison_df['MLP_Pred'] = (comparison_df['MLP_Prob'] > 0.5).astype(int)
comparison_df['Attn_V1_Pred'] = (comparison_df['Attention_V1_Prob'] > 0.5).astype(int)
comparison_df['Attn_V2_Pred'] = (comparison_df['Attention_V2_Prob'] > 0.5).astype(int)

# 3. Lưu thành một file duy nhất
comparision_df

NameError: name 'y_prob_mlp' is not defined